[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/purple/notebooks/purple_chemical_space.ipynb)

# Exploring the chemical space of the HIV-1 dataset

**Purple group · HIV**

The models in the previous notebook were trained on twenty thousand molecules without
ever looking at what those molecules are. This notebook turns them into a map, using
coordinates from the Ersilia model `eos1klk`, and asks the question that decides whether
the models are any use to this project: are the natural products the group wants to
screen anywhere near the chemistry the models were trained on?

## What you will do

- Load the curated HIV-1 dataset together with the chemical space coordinates from `eos1klk`
- Plot how the measured potency is distributed
- Draw the chemical space and colour it by activity
- Find the most common scaffolds, and see where chalcones sit among them

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "purple"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The dataset and its coordinates

Two files go into this notebook.

`data/hiv1_curated.csv` is what came out of the curation notebook: one row per molecule,
with its potency as a **pActivity** (the higher the number, the more potent the molecule)
and an `activity` label, 1 for active and 0 for inactive, using a cutoff of 1 micromolar
(pActivity 6).

`data/eos1klk_hiv1_curated.csv` is the output of the Ersilia model
[eos1klk](https://github.com/ersilia-os/eos1klk). The model takes a molecule and returns
eight numbers: a pair of coordinates on each of four different maps. The maps were built
from the Ersilia Reference Library, 1.3 million compounds, so a molecule's position says
where it sits among molecules in general, not only among the ones in this dataset.

> **Note:** this is how that second file was made, and how you make one for
> any other set of molecules. Take the one-column SMILES file the curation notebook
> saved (`hiv1_curated_smiles.csv`), run it through `eos1klk` in Ersilia, and put the
> result in the group's Drive folder **Projects/PurpleTeam/Data**, named
> `eos1klk_<what the molecules are>.csv`. It then gets copied into `data/` here, and the
> notebook can read it. Nothing in this notebook needs a GPU or an internet connection
> once the file exists.

Load both files and join them, molecule by molecule, on the SMILES.

In [ ]:
import pandas as pd
import stylia
from scripts import chemspace, modelling

THRESHOLD = 6.0  # pActivity 6 is the 1 micromolar cutoff used to label the molecules

space = chemspace.load_space("data/hiv1_curated.csv", "data/eos1klk_hiv1_curated.csv")
print(f"{len(space):,} molecules, each with coordinates on four maps")
space[["chembl_id", "smiles", "pactivity", "activity"]].head()

These are the eight coordinate columns. Each pair is one map: `pca_x` and
`pca_y` place the molecule on the PCA map, `umap_x` and `umap_y` on the UMAP map, and so
on. The numbers have no units and mean nothing on their own; only the distance between
two molecules does.

In [ ]:
space.filter(regex="_(x|y)$").head()

## 2. How the measurements are spread

A map is only worth reading if you know what is being mapped. Before plotting any
chemistry, look at the numbers themselves: how potent the molecules are, and how many
of them end up on each side of the cutoff.

Set the plotting style once. Every plot in this notebook uses `stylia`, so
they all come out with the same fonts and colours.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()  # nc.purple, nc.mint, nc.gray, nc.pink, nc.plum ...

print("Plots in this notebook use the Ersilia style, in the purple group's colour.")

A histogram cuts the pActivity range into bins and counts how many
molecules fall in each one. The dashed line is the cutoff that separates actives from
inactives.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(space["pactivity"], bins=60, range=(3, 11.5), color=nc.purple)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Measured potency (median {space['pactivity'].median():.2f})")

The cutoff falls near the middle of the distribution, which is why the two
classes came out almost the same size. Count them to confirm.

In [ ]:
counts = space["activity"].value_counts().rename({0: "inactive", 1: "active"})
counts = counts.reindex(["inactive", "active"])  # the order the colours below assume

fig, axs = stylia.create_figure(1, 1, width=0.5)
ax = axs.next()
ax.bar(counts.index, counts.values, width=0.4, color=[nc.purple, nc.mint])
ax.set_xlim(-0.6, 1.6)  # room around the bars, so two categories do not fill the panel
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

## 3. The map of chemical space

A molecule has thousands of properties, so it cannot be drawn on a page as it is. A
projection squeezes all of them into two numbers, keeping molecules that are alike close
together. `eos1klk` gives four of them, and they disagree on purpose:

- **PCA** is the plainest: it keeps the big distances honest, so far-apart points really
  are different, but everything piles up in the middle.
- **t-SNE** pulls neighbours together and separates groups clearly. The distance between
  two separate clusters means nothing.
- **UMAP** does something similar but keeps a little more of the overall shape.
- **TMAP** lays the molecules out as a tree, which spreads out sparse regions.

None of them has a right answer. Read them together: a group of molecules that stays
together in all four is a real family.

> **Note:** there are more than twenty thousand molecules here, so the points are drawn
> almost transparent. A dark patch is not one molecule, it is hundreds stacked up.

Start with UMAP, one point per molecule.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_points(ax, space, "umap", color=nc.purple)
chemspace.label_space(ax, "umap", f"{len(space):,} curated HIV-1 molecules")

Now the same molecules on all four maps, side by side.

In [ ]:
fig, axs = stylia.create_figure(2, 2, width=1.0, height=1.0)  # a square figure
for projection, letter in zip(chemspace.PROJECTIONS, "ABCD"):
    ax = axs.next()
    chemspace.plot_points(ax, space, projection, color=nc.purple, alpha=0.2)
    ax.set_box_aspect(1)  # each panel is a square, whatever range its numbers cover
    chemspace.label_space(ax, projection, chemspace.PROJECTIONS[projection], abc=letter)

> **Exercise:** pick a map and describe what you see. How many separate
> clumps are there? Does the same number of clumps show up in the other three maps? The
> rest of this notebook uses UMAP, but every cell below works with `"pca"`, `"tsne"` or
> `"tmap"` instead.

## 4. Potency on the map

The map so far only shows chemistry. Colouring it by what was measured is what makes it
useful: if the potent molecules sit in one part of the map, chemistry and potency go
together, and a model has something to learn. If they are scattered everywhere, small
changes matter more than the overall shape of the molecule.

First the two classes, active and inactive, in two colours.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for value, color, name in [(0, nc.purple, "inactive"), (1, nc.mint, "active")]:
    chemspace.plot_points(ax, space[space["activity"] == value], "umap", color=color, label=name)
ax.legend()
chemspace.label_space(ax, "umap", "Active and inactive molecules")

Now the potency itself, as a colour that goes from pale to dark.

In [ ]:
cm = stylia.FadingColormap("plum")
cm.fit(space["pactivity"])

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_points(ax, space, "umap", color=cm.transform(space["pactivity"]), alpha=0.4)
chemspace.label_space(ax, "umap", "Darker means more potent")

> **Exercise:** look for a region of the map where the dark points are
> packed together, and one where dark and pale points are mixed. Use the cell below to
> read off the molecules in a region you choose: change the four numbers to the corner
> coordinates of the box you are interested in. The box already in the cell is one of
> the potent regions; for contrast, try `(-0.60, -0.35)` and `(0.10, 0.30)`, which is
> the corner section 7 comes back to.

A box drawn by hand. `x_range` and `y_range` are the left-right and
bottom-top edges of the region to look inside.

In [ ]:
x_range, y_range = (-0.30, 0.00), (-0.55, -0.30)  # change these

x, y = chemspace.coordinates(space, "umap")
inside = (x > x_range[0]) & (x < x_range[1]) & (y > y_range[0]) & (y < y_range[1])
print(f"{inside.sum():,} molecules in the box | "
      f"{space.loc[inside, 'activity'].mean():.0%} active | "
      f"median pActivity {space.loc[inside, 'pactivity'].median():.2f}")
chemspace.draw_molecules(space.loc[inside, "smiles"].head(4),
                         space.loc[inside, "pactivity"].head(4).round(2))

## 5. Where the approved HIV drugs sit

Twenty-six molecules are approved medicines for HIV, and all of them are somewhere in
this dataset: they were measured in the same kinds of assay as everything else. They are
the closest thing we have to a set of known right answers, so where they land is worth
looking at carefully.

`data/hiv_drugs_approved.csv` holds their names and structures, taken from ChEMBL. Each
one is matched to the dataset by its **InChIKey**, the code that identifies a molecule
exactly, so no drug is matched to a near-relative by mistake.

Draw the whole dataset by class, faintly, and put the drugs on top as
large points.

In [ ]:
drugs = chemspace.find_drugs(space, "data/hiv_drugs_approved.csv")

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for value, color, name in [(0, nc.gray, "inactive"), (1, nc.mint, "active")]:
    chemspace.plot_points(ax, space[space["activity"] == value], "umap",
                          color=color, label=name, alpha=0.25)
ax.scatter(*chemspace.coordinates(drugs, "umap"), color=nc.purple,
           s=stylia.MARKERSIZE_BIG, label=f"approved drug ({len(drugs)})")
for handle in ax.legend().legend_handles:
    handle.set_alpha(1)  # otherwise the faint points make a legend nobody can read
chemspace.label_space(ax, "umap", "Where the approved HIV drugs sit")

And the numbers behind those large points, most potent first.

In [ ]:
drugs[["drug", "chembl_id", "pactivity", "activity"]].sort_values(
    "pactivity", ascending=False).reset_index(drop=True)

Twenty-four of the twenty-six are labelled active. The two that are not
are worth knowing about: **abacavir** and **didanosine** are prodrugs, inactive as
supplied, and the cell has to convert them before they block anything. The assay measured
the molecule that went into the well, not the one that does the work. An `inactive` label
describes an experiment, not a compound.

The drugs are scattered right across the map rather than sitting in one
place, which is what you would expect from molecules that attack five different steps of
the virus life cycle. It is worth asking whether they sit in the crowded parts of the map
or the thin ones, because that is what decides how much a model has to learn from near
each of them. Counting neighbours answers it.

In [ ]:
crowding = pd.Series(chemspace.neighbour_counts(space, "umap"))
per_drug = pd.Series(chemspace.neighbour_counts(space, "umap", points=drugs),
                     index=drugs["drug"])
print(f"molecules within 0.05 on the map: median {crowding.median():.0f} across the "
      f"dataset, {per_drug.median():.0f} around the approved drugs")
per_drug.sort_values().rename("neighbours").head(6).to_frame()

This is the opposite of what you might guess. The drugs sit in **less**
crowded neighbourhoods than the average molecule in the dataset, and some of the newest
and most potent of them sit in the emptiest. Efavirenz has no neighbour at all within
that distance; dolutegravir, bictegravir and cabotegravir have a handful each.

The crowded regions are not where the drugs are. They are series of analogues: hundreds
of variations made around one idea, most of them never going anywhere. A drug is often a
molecule nobody else had made.

That matters for the rest of this notebook. A molecule sitting in a thin part of the map
is not a bad molecule, as three of the best drugs here show. What it means is that a model
has very little to go on there, so its prediction rests on much less evidence.

> **Exercise:** pick three drugs from different classes, for example
> `Zidovudine`, `Ritonavir` and `Dolutegravir`, and find each one on the map. Do they sit
> together or far apart? They attack three different steps of the virus life cycle, so
> what would it mean if the map had put them in the same place?

## 6. The scaffolds behind the clusters

A **scaffold** is what is left of a molecule when every side chain is removed: its rings
and the bits that join them. Two molecules with the same scaffold are variations on one
idea, usually from the same paper or the same medicinal chemistry programme. Scaffolds
are the quickest way to find out what the clumps on the map actually are.

This is the same `murcko_scaffolds` the modelling notebook used to build its scaffold
split, so the groups on this map are exactly the groups the model was tested across.

Work out the scaffold of every molecule and count the most common ones. A
molecule with no rings at all has no scaffold, so it stands in for itself and counts as
a series of one.

In [ ]:
space["scaffold"] = modelling.murcko_scaffolds(space["smiles"])
common = space["scaffold"].value_counts()
print(f"{len(common):,} different scaffolds | "
      f"{(common == 1).sum():,} of them cover a single molecule")
common.head(6).rename("molecules")

This is what those six look like, numbered from the most common one down.

In [ ]:
top = common.head(6)
chemspace.draw_molecules(top.index, [f"{i}: {n} molecules" for i, n in enumerate(top.values, 1)],
                         per_row=3)

Now put them on the map. Each scaffold keeps the number it has in the
drawing above, and everything else stays grey.

In [ ]:
palette = stylia.CategoricalPalette("ersilia").get(len(top))

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
for rank, ((scaffold, size), color) in enumerate(zip(top.items(), palette), 1):
    chemspace.plot_points(ax, space[space["scaffold"] == scaffold], "umap",
                          color=color, label=f"{rank} ({size})", alpha=0.9)
ax.legend()
chemspace.label_space(ax, "umap", "The six most common scaffolds")

> **Exercise:** scaffold 1 is the five-fused-ring skeleton of **betulinic
> acid**, a natural product from birch bark whose derivatives block the last step of HIV
> assembly. Find it on the map. Does it sit on its own, or among the synthetic molecules?
> A natural product series that sits apart from everything else is the clearest possible
> warning about where a model trained on this data can and cannot be trusted.

## 7. Where do the chalcones sit?

This is the section to fill in for your own project. The group wants to screen African
natural products, and **chalcones** are one of the classes it is interested in: two
aromatic rings joined by a short unsaturated bridge, found in many plants and reported
against a range of targets including HIV.

Before predicting anything, check whether molecules like those are in the training data
at all: a model asked about a kind of chemistry it has never seen will still return a
number, and that number will be guesswork.

A ring system is written as a **SMARTS** pattern, a short piece of text describing a
partial molecule that can be searched for inside a bigger one. Chalcones and two
relatives are filled in below. Add your own, then re-run the cells that follow.

Write the ring systems your group cares about here, as `name: SMARTS`.
`cC(=O)C=Cc` is the chalcone bridge: an aromatic ring, a carbonyl, a double bond, and a
second aromatic ring.

In [ ]:
PATTERNS = {
    "chalcone": "cC(=O)C=Cc",
    "dihydrochalcone": "cC(=O)CCc",
    "flavone": "O=c1cc(-c2ccccc2)oc2ccccc12",
    # "coumarin": "O=c1ccc2ccccc2o1",
    # "your scaffold": "...",
}

found = {name: int(chemspace.has_substructure(space["smiles"], smarts).sum())
         for name, smarts in PATTERNS.items()}
pd.Series(found).rename("molecules in the dataset")

Mark them on the map. Molecules matching none of the patterns stay grey,
so the coloured points are the ones to look at.

> **Note:** a molecule can contain more than one of these ring systems, so it is given
> the first pattern in the list that it matches. That is why a count in the legend can
> be one or two lower than the count in the table above.

In [ ]:
space["pattern"] = chemspace.label_substructures(space["smiles"], PATTERNS)
colors = dict(zip(PATTERNS, stylia.CategoricalPalette("ersilia").get(len(PATTERNS))))

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
chemspace.plot_backdrop(ax, space, "umap")
for name, color in colors.items():
    subset = space[space["pattern"] == name]
    chemspace.plot_points(ax, subset, "umap", color=color, label=f"{name} ({len(subset)})", alpha=0.9)
ax.legend()
chemspace.label_space(ax, "umap", "Ring systems the group is interested in")

And how potent the matching molecules are, compared with everything else.

In [ ]:
summary = space.groupby("pattern").agg(molecules=("smiles", "size"),
                                       median_pactivity=("pactivity", "median"),
                                       percent_active=("activity", "mean"))
summary["percent_active"] = (summary["percent_active"] * 100).round(0)
summary.sort_values("molecules", ascending=False)

> **Exercise:** the chalcones in this dataset are far less often active than
> the dataset as a whole. Write down, in one sentence each: how many chalcones the models
> have seen, whether those were mostly actives or mostly inactives, and where they sit on
> the map. Then ask the harder question. If a model is trained on a set where nearly
> every chalcone is inactive, what will it predict for a chalcone it has never seen, and
> is that a real prediction or just the model repeating what it was shown?

> **Note:** to put your own natural products on this map, write their
> SMILES into a one-column file, run it through `eos1klk`, put the output in Drive next
> to the file this notebook used, and plot the two sets together, the curated molecules
> as the grey backdrop and yours on top.

## Summary

- You joined the curated HIV-1 dataset to the coordinates from the Ersilia model
  `eos1klk` and drew the group's chemical space four different ways.
- The measured potencies spread over seven orders of magnitude, and the 1 micromolar
  cutoff falls near the middle, which is why the two classes came out balanced.
- All twenty-six approved HIV drugs are in the dataset. They are spread across the
  whole map, and they sit in thinner neighbourhoods than the average molecule: the
  crowded regions are series of analogues, not the drugs themselves.
- The dataset is made of a few thousand molecule families. The largest is the
  betulinic acid skeleton, a natural product series, and it occupies its own region of
  the map.
- Marking chalcones showed how little of that chemistry the dataset contains, and how
  rarely it is active, which is what decides whether the models from the previous
  notebook can say anything useful about the group's natural products.

**Next:** run the group's own natural product list through `eos1klk` and plot it on this
map, then score the same molecules with the models from `purple_baseline_models.ipynb`
and treat any prediction that lands far from the grey points with suspicion.